In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
import kagglehub

path = kagglehub.dataset_download("saurabh00007/diabetescsv")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os
df = pd.read_csv(os.path.join(path, "diabetes.csv"))
df.head(800)

In [ ]:
df.info()
df.shape
df.describe()

In [ ]:
df['Outcome'].value_counts()

In [ ]:
sns.countplot(x='Outcome', data=df)
plt.title("diabetes distribution ")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True,cmap='coolwarm')
plt.show()

In [ ]:
sns.boxplot(x='Outcome',y='Glucose',data=df)
plt.show()

In [ ]:
sns.histplot(df['BMI'],kde=True)
plt.show()

In [ ]:
sns.pairplot(df,hue='Outcome')
plt.show()

In [ ]:
q1=df.quantile(0.25)
q3=df.quantile(0.75)
iqr=q3-q1
df=df[~((df<(q1-1.5*iqr))|(df>(q3+1.5*iqr))).any(axis=1)]

In [ ]:
# df['BMI_Category']=pd.cut()

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
log_reg = models["Logistic Regression"]
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log_reg))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_log_reg))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log_reg))

In [ ]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)


In [ ]:
# do smote

In [ ]:
from sklearn.metrics import accuracy_score, precision_score,recall_score,f1_score
import pandas as pd
models={
    "Logistic Regression":LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree":DecisionTreeClassifier(random_state=42),
    "Random Forest":RandomForestClassifier(random_state=42),
    "SVM":SVC(random_state=42, probability=True),
    "KNN":KNeighborsClassifier()
}

In [ ]:
results=[]
for name,model in models.items():
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    acc=accuracy_score(y_test,y_pred)
    prec=precision_score(y_test,y_pred)
    rec=recall_score(y_test,y_pred)
    f1=f1_score(y_test,y_pred)

    results.append(
        {
            "Model":name,
            "Accuracy":acc,
            "Precision":prec,
            "Recall":rec,
            "F1 Score":f1
        }
    )


In [ ]:
results_df=pd.DataFrame(results)
print(results_df)

In [ ]:
plt.bar(results_df["Model"],results_df["Accuracy"])
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.xticks(rotation=20)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve,auc
import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
for name,model in models.items():
  if name in ["Logistic Regression", "SVM", "KNN"]:
    model.fit(X_train_scaled, y_train)
    if hasattr(model,"predict_proba"):
      y_prob=model.predict_proba(X_test_scaled)[:,1]
    else:
      y_prob=model.decision_function(X_test_scaled)
  else:
    model.fit(X_train,y_train)
    if hasattr(model,"predict_proba"):
      y_prob=model.predict_proba(X_test)[:,1]
    else:
      y_prob=model.decision_function(X_test)

  fpr,tpr,_=roc_curve(y_test,y_prob)
  roc_auc=auc(fpr,tpr)
  plt.plot(fpr,tpr,label=f"{name} (AUC={roc_auc:.2f})")
  plt.xlabel("False Positive Rate")
  plt.ylabel("True Positive Rate")
  plt.title("ROC Curve")
  plt.legend()
plt.show()

In [ ]:
best_model_name=results_df.loc[results_df["Accuracy"].idxmax(),"Model"]
print("Best model: ",best_model_name)

In [ ]:
best_model=models[best_model_name]
best_model.fit(X_train,y_train)
pred_best_model=best_model.predict(X_test)

In [ ]:
cm=confusion_matrix(y_test,pred_best_model)
sns.heatmap(cm,annot=True,fmt="d")
plt.title("Confusion matrix for " + best_model_name)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
print(classification_report(y_test,pred_best_model))

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [ ]:
from imblearn.over_sampling import SMOTE
smote=SMOTE(random_state=42)
X_train,y_train=smote.fit_resample(X_train,y_train)

In [ ]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [ ]:
cm=confusion_matrix(y_test,pred_best_model)
sns.heatmap(cm,annot=True,fmt="d")
plt.title("Confusion matrix for " + best_model_name)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
print(classification_report(y_test,pred_best_model))

In [ ]:
import pickle
pickle.dump(best_model,open("diabetes_model.pkl","wb"))
pickle.dump(scaler,open("scaler.pkl","wb"))

In [ ]:
import gradio as gr
import pickle
import numpy as np
import pandas as pd
model=pickle.load(open("diabetes_model.pkl","rb"))
scaler=pickle.load(open("scaler.pkl","rb"))

def predict(glucose,bp,bmi,age,pregencies):
  data=np.array([[glucose,bp,bmi,age]])
  data=scaler.transform(data)
  prediction=model.predict(data)
  result=model.predict(data)
  return "Diabetic" if result==1 else "Non-Diabetic"

demo=gr.Interface(
    fn=predict,
    inputs=["number","number","number","number","number"],
    outputs="text",
    title="Diabetes Prediction App",

)
demo.launch( share=True)

In [ ]:
import gradio as gr
import pickle
import numpy as np
import pandas as pd

# Load the trained model and scaler
best_model = pickle.load(open("diabetes_model.pkl", "rb"))
scaler = pickle.load(open("scaler.pkl", "rb"))

# Define the prediction function
def predict_diabetes(
    Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI,
    DiabetesPedigreeFunction, Age
):
    input_data = pd.DataFrame([[Pregnancies, Glucose, BloodPressure, SkinThickness,
                                  Insulin, BMI, DiabetesPedigreeFunction, Age]],
                                columns=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                                         'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'])

    scaled_input_data = scaler.transform(input_data)

    prediction = best_model.predict(scaled_input_data)

    if prediction[0] == 1:
        return "Diabetic"
    else:
        return "Non-Diabetic"

interface = gr.Interface(
    fn=predict_diabetes,
    inputs=[
        gr.Number(label="Pregnancies"),
        gr.Number(label="Glucose"),
        gr.Number(label="BloodPressure"),
        gr.Number(label="SkinThickness"),
        gr.Number(label="Insulin"),
        gr.Number(label="BMI"),
        gr.Number(label="DiabetesPedigreeFunction"),
        gr.Number(label="Age")
    ],
    outputs="text",
    title="Diabetes Prediction App",
    description="Enter patient details to predict if they have diabetes."
)

interface.launch(inline=True, share=True)